# Run-Aggregate Statistik (llm-01)

Zwei Analysen auf Basis der `run_aggregate.json` der `state-presentation-llm-01` Runs:

**Analyse 1 – Stabilitaet der Sonnet-Runs:** Drei Wiederholungen desselben Modells
(`sonnet4-5`) werden verglichen, um zu messen, wie reproduzierbar die Scores sind
(Streuung der Run-Mittelwerte, paarweiser MAE, Pro-Datei-Streuung ueber die Runs).

**Analyse 2 – Modellvergleich:** Vergleich verschiedener Modelle (`sonnet`, `gpt`, `qwen`)
anhand derselben Kennzahlen. Laeuft erst vollstaendig, sobald fuer alle Modelle ein
`run_aggregate.json` vorliegt (Qwen-Run ggf. noch nicht abgeschlossen).

In [ ]:
import json
import statistics as st
from collections import Counter
from itertools import combinations
from pathlib import Path

# Repo-Root relativ zum Notebook (playground/notebooks -> ../../)
REPO_ROOT = Path.cwd().parents[1]
RUNS_DIR = REPO_ROOT / "run_outputs"

# Analyse 1: drei Wiederholungen desselben Modells (sonnet4-5)
SONNET_RUNS = [
    'run_2026-06-12_13-21-59_state-presentation-llm-01_sonnet4-5',
    'run_2026-06-15_07-55-43_state-presentation-llm-01_sonnet4-5',
    'run_2026-06-15_08-11-47_state-presentation-llm-01_sonnet4-5',
]

# Analyse 2: je ein Run pro Modell. Fehlt fuer ein Modell das run_aggregate.json,
# ueberspringt die Analyse es automatisch.
MODEL_RUNS = {
    'sonnet4-5': 'run_2026-06-12_13-21-59_state-presentation-llm-01_sonnet4-5',
    'gpt4-5':    'run_2026-06-12_13-26-51_state-presentation-llm-01_gpt4-5',
    'qwen3-5':   'run_2026-06-15_07-38-53_state-presentation-llm-01_sample_qwen3-5',
}

for run_name in SONNET_RUNS:
    print(run_name)

In [ ]:
def load_aggregate(run) -> dict:
    # akzeptiert einen Run-Namen (str) ODER einen Path; loest Namen relativ zu RUNS_DIR auf
    run_dir = run if isinstance(run, Path) else RUNS_DIR / run
    return json.loads((run_dir / "run_aggregate.json").read_text())


def compute_stats(agg: dict) -> dict:
    files = agg["files"]
    scores = [f["overall_score"] for f in files.values()]
    pass_rates = [f["overall_pass_rate"] for f in files.values()]
    grades = Counter(f["quality_grade"] for f in files.values())
    return {
        "timestamp": agg["timestamp"],
        "n": len(scores),
        "mean": st.mean(scores),
        "stdev": st.stdev(scores),
        "pstdev": st.pstdev(scores),
        "median": st.median(scores),
        "min": min(scores),
        "max": max(scores),
        "range": max(scores) - min(scores),
        "cv": st.stdev(scores) / st.mean(scores),  # Variationskoeffizient
        "pass_rate_mean": st.mean(pass_rates),
        "pass_rate_stdev": st.stdev(pass_rates),
        "grades": dict(sorted(grades.items())),
        "scores": scores,
    }


stats = [compute_stats(load_aggregate(p)) for p in SONNET_RUNS]
for s in stats:
    print(f"\n=== {s['timestamp']}  (n={s['n']}) ===")
    print(f"  mean           : {s['mean']:.4f}")
    print(f"  stdev (sample) : {s['stdev']:.4f}")
    print(f"  pstdev (pop.)  : {s['pstdev']:.4f}")
    print(f"  median         : {s['median']:.4f}")
    print(f"  min / max      : {s['min']:.4f} / {s['max']:.4f}")
    print(f"  range          : {s['range']:.4f}")
    print(f"  CV             : {s['cv']:.4f}")
    print(f"  pass_rate mean : {s['pass_rate_mean']:.4f} (stdev {s['pass_rate_stdev']:.4f})")
    print(f"  grades         : {s['grades']}")

In [ ]:
# Analyse 1a: Kennzahlen der Sonnet-Runs nebeneinander + Streuung ueber die Runs.
metrics = ["mean", "stdev", "median", "min", "max", "range", "cv", "pass_rate_mean"]
labels = [s["timestamp"] for s in stats]

print(f"{'Metrik':<16}" + "".join(f"{l:>22}" for l in labels) + f"{'spread':>10}")
print("-" * (16 + 22 * len(stats) + 10))
for m in metrics:
    vals = [s[m] for s in stats]
    spread = max(vals) - min(vals)            # max-min ueber die Runs
    print(f"{m:<16}" + "".join(f"{v:>22.4f}" for v in vals) + f"{spread:>10.4f}")

# Reproduzierbarkeit auf Run-Ebene: wie stark schwankt der Gesamt-Mittelwert?
run_means = [s["mean"] for s in stats]
print(f"\nMittelwert der Run-Means : {st.mean(run_means):.4f}")
print(f"Stdev der Run-Means      : {st.stdev(run_means):.4f}")
print(f"Spannweite der Run-Means : {max(run_means) - min(run_means):.4f}")

In [ ]:
# Analyse 1b: Wie stark unterscheiden sich die Sonnet-Runs Datei fuer Datei?
# Gleicher Mittelwert heisst nicht gleiche Einzel-Scores - das misst hier den echten Unterschied.

files_per_run = [load_aggregate(r)["files"] for r in SONNET_RUNS]
common = sorted(set.intersection(*(set(f) for f in files_per_run)))


def pair_stats(fa, fb):
    d = [fb[f]["overall_score"] - fa[f]["overall_score"] for f in common]
    xa = [fa[f]["overall_score"] for f in common]
    xb = [fb[f]["overall_score"] for f in common]
    ma, mb = st.mean(xa), st.mean(xb)
    cov = sum((x - ma) * (y - mb) for x, y in zip(xa, xb))
    den = (sum((x - ma) ** 2 for x in xa) * sum((y - mb) ** 2 for y in xb)) ** 0.5
    return {
        "mae": st.mean(abs(x) for x in d),       # typische Abweichung pro Datei
        "bias": st.mean(d),                       # systematischer Trend (B - A)
        "pearson": cov / den if den else float("nan"),
    }


# (1) paarweise zwischen je zwei Runs
print(f"Gemeinsame Dateien: {len(common)}\n")
print(f"{'Run-Paar':<12}{'MAE':>10}{'Bias':>10}{'Pearson':>10}")
print("-" * 42)
maes = []
for i, j in combinations(range(len(SONNET_RUNS)), 2):
    ps = pair_stats(files_per_run[i], files_per_run[j])
    maes.append(ps["mae"])
    print(f"{f'Run {i + 1} vs {j + 1}':<12}{ps['mae']:>10.4f}{ps['bias']:>+10.4f}{ps['pearson']:>10.4f}")
print(f"\nDurchschnittlicher paarweiser MAE: {st.mean(maes):.4f}")

# (2) pro Datei: Streuung ueber alle Runs (direktes Reproduzierbarkeits-Mass)
per_file = {f: [fr[f]["overall_score"] for fr in files_per_run] for f in common}
file_stdevs = {f: st.stdev(v) for f, v in per_file.items()}
file_ranges = {f: max(v) - min(v) for f, v in per_file.items()}
print(f"\nPro-Datei Streuung ueber {len(SONNET_RUNS)} Runs:")
print(f"  mittlere Stdev pro Datei : {st.mean(file_stdevs.values()):.4f}")
print(f"  max Stdev pro Datei      : {max(file_stdevs.values()):.4f}")
print(f"  mittlere Spannweite      : {st.mean(file_ranges.values()):.4f}")

print("\nUnstabilste Dateien (groesste Spannweite ueber die Runs):")
for f in sorted(common, key=lambda f: file_ranges[f], reverse=True)[:8]:
    vals = " / ".join(f"{x:.3f}" for x in per_file[f])
    print(f"  {f:<44} [{vals}]  range={file_ranges[f]:.3f}")

In [ ]:
import re

# Varianz zwischen den Datensaetzen (pro Sonnet-Run).
# Jeder Datensatz (z.B. geo_gdi-de) hat mehrere Wiederholungen (_02, _03).
# - within-Varianz : Streuung der Wiederholungen INNERHALB eines Datensatzes
# - between-Varianz: Streuung der Datensatz-Mittelwerte ZWISCHEN den Datensaetzen


def dataset_variance(agg: dict) -> dict:
    groups: dict[str, list[float]] = {}
    for fname, f in agg["files"].items():
        base = re.sub(r"(_\d+)?\.rdf$", "", fname)  # _02/_03-Suffix entfernen
        groups.setdefault(base, []).append(f["overall_score"])

    per_dataset = {
        base: {
            "n": len(sc),
            "mean": st.mean(sc),
            "variance": st.pvariance(sc) if len(sc) > 1 else 0.0,
            "stdev": st.pstdev(sc) if len(sc) > 1 else 0.0,
        }
        for base, sc in sorted(groups.items())
    }
    means = [d["mean"] for d in per_dataset.values()]
    within = [d["variance"] for d in per_dataset.values() if d["n"] > 1]
    return {
        "per_dataset": per_dataset,
        "n_datasets": len(per_dataset),
        "between_variance": st.pvariance(means),  # Varianz der Datensatz-Means
        "mean_within_variance": st.mean(within) if within else 0.0,
    }


for p, s in zip(SONNET_RUNS, stats):
    dv = dataset_variance(load_aggregate(p))
    print(f"\n=== {s['timestamp']}  ({dv['n_datasets']} Datensaetze) ===")
    for base, d in dv["per_dataset"].items():
        print(f"  {base:<38} mean={d['mean']:.4f}  within-var={d['variance']:.5f}  (σ={d['stdev']:.4f})")
    print(f"  -> between-dataset variance : {dv['between_variance']:.5f}")
    print(f"  -> mean within-dataset var  : {dv['mean_within_variance']:.5f}")

In [ ]:
# Verteilung der scores ueber die Sonnet-Runs visualisieren
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
for s in stats:
    ax.hist(s["scores"], bins=10, alpha=0.5, label=f"{s['timestamp']} (σ={s['stdev']:.3f})")
ax.set_xlabel("overall_score")
ax.set_ylabel("Anzahl Dateien")
ax.set_title(f"Verteilung overall_score – {len(stats)} Sonnet-Runs")
ax.legend()
plt.tight_layout()
plt.show()

## Analyse 2 – Modellvergleich (sonnet / gpt / qwen)

Dieselben Kennzahlen wie in Analyse 1, aber pro **Modell** statt pro Wiederholung.
Modelle ohne vorhandenes `run_aggregate.json` (z.B. ein noch laufender Qwen-Run)
werden uebersprungen und gemeldet, statt einen Fehler auszuloesen.

In [ ]:
# Pro-Modell-Kennzahlen. Laedt nur Modelle mit vorhandenem run_aggregate.json.
model_stats = {}
for model, run in MODEL_RUNS.items():
    if not (RUNS_DIR / run / "run_aggregate.json").exists():
        print(f"[skip] {model}: kein run_aggregate.json ({run})")
        continue
    model_stats[model] = compute_stats(load_aggregate(run))

models = list(model_stats)
metrics = ["mean", "stdev", "median", "min", "max", "range", "cv", "pass_rate_mean"]

print(f"\n{'Metrik':<16}" + "".join(f"{m:>14}" for m in models))
print("-" * (16 + 14 * len(models)))
for me in metrics:
    print(f"{me:<16}" + "".join(f"{model_stats[m][me]:>14.4f}" for m in models))

for m in models:
    print(f"\n{m:<10} grades: {model_stats[m]['grades']}")

In [ ]:
# Paarweiser Modellvergleich Datei fuer Datei (nur fuer geladene Modelle).
# Zeigt, wie unterschiedlich die Modelle dieselben Datensaetze bewerten.
model_files = {m: load_aggregate(MODEL_RUNS[m])["files"] for m in models}

if len(models) >= 2:
    common_m = sorted(set.intersection(*(set(f) for f in model_files.values())))
    print(f"Gemeinsame Dateien: {len(common_m)}\n")
    print(f"{'Modell-Paar':<22}{'MAE':>10}{'Bias':>10}{'Pearson':>10}")
    print("-" * 52)
    for a, b in combinations(models, 2):
        fa, fb = model_files[a], model_files[b]
        d = [fb[f]["overall_score"] - fa[f]["overall_score"] for f in common_m]
        xa = [fa[f]["overall_score"] for f in common_m]
        xb = [fb[f]["overall_score"] for f in common_m]
        ma, mb = st.mean(xa), st.mean(xb)
        cov = sum((x - ma) * (y - mb) for x, y in zip(xa, xb))
        den = (sum((x - ma) ** 2 for x in xa) * sum((y - mb) ** 2 for y in xb)) ** 0.5
        pearson = cov / den if den else float("nan")
        print(f"{f'{a} vs {b}':<22}{st.mean(abs(x) for x in d):>10.4f}"
              f"{st.mean(d):>+10.4f}{pearson:>10.4f}")
else:
    print("Mindestens 2 Modelle mit Aggregate noetig fuer den paarweisen Vergleich.")